In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
digits = load_digits()
X = digits.images.astype(np.float64) / 16.0   # 原像素范围 0~16
y = digits.target

X = X[:, None, :, :]   # (N, 1, 8, 8)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

encoder = OneHotEncoder(sparse_output=False)
y_train_onehot = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_onehot = encoder.transform(y_test.reshape(-1, 1))

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

In [ ]:
def im2col_single(x, kH, kW, stride=1, padding=0):
    # x: (C, H, W)
    C, H, W = x.shape
    x_pad = np.pad(x, ((0,0), (padding,padding), (padding,padding)), mode="constant")
    H_pad, W_pad = x_pad.shape[1], x_pad.shape[2]

    out_h = (H_pad - kH) // stride + 1
    out_w = (W_pad - kW) // stride + 1

    cols = []
    for i in range(out_h):
        for j in range(out_w):
            patch = x_pad[:, i*stride:i*stride+kH, j*stride:j*stride+kW]
            cols.append(patch.reshape(-1))
    cols = np.array(cols).T  # (C*kH*kW, out_h*out_w)
    return cols, out_h, out_w


def softmax(logits):
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / np.sum(exp, axis=1, keepdims=True)


def cross_entropy_loss(probs, y_onehot):
    eps = 1e-12
    return -np.mean(np.sum(y_onehot * np.log(probs + eps), axis=1))


def accuracy_from_logits(logits, y_true):
    pred = np.argmax(logits, axis=1)
    return np.mean(pred == y_true)

In [ ]:
class Conv2D:
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=0, weight_scale=0.1):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * weight_scale
        self.b = np.zeros(out_channels)

    def forward(self, x):
        # x: (N, C, H, W)
        self.x = x
        N, C, H, W = x.shape
        k = self.kernel_size

        cols_list = []
        out_list = []

        for n in range(N):
            cols, out_h, out_w = im2col_single(
                x[n], k, k, stride=self.stride, padding=self.padding
            )
            cols_list.append(cols)

            W_row = self.W.reshape(self.out_channels, -1)  # (F, C*k*k)
            out = W_row @ cols + self.b[:, None]          # (F, out_h*out_w)
            out_list.append(out.reshape(self.out_channels, out_h, out_w))

        self.cols_list = cols_list
        self.out_h = out_h
        self.out_w = out_w

        return np.stack(out_list, axis=0)

    def backward(self, dout):
        # dout: (N, F, out_h, out_w)
        N, F, out_h, out_w = dout.shape
        C = self.in_channels
        k = self.kernel_size

        dW = np.zeros_like(self.W)
        db = np.zeros_like(self.b)
        dx = np.zeros_like(self.x)

        W_row = self.W.reshape(F, -1)

        for n in range(N):
            cols = self.cols_list[n]  # (C*k*k, out_h*out_w)
            dout_n = dout[n].reshape(F, -1)

            dW += (dout_n @ cols.T).reshape(self.W.shape)
            db += np.sum(dout_n, axis=1)

            dcols = W_row.T @ dout_n  # (C*k*k, out_h*out_w)

            x_pad = np.pad(
                np.zeros_like(self.x[n]),
                ((0,0), (self.padding,self.padding), (self.padding,self.padding)),
                mode="constant"
            )

            idx = 0
            for i in range(out_h):
                for j in range(out_w):
                    patch_grad = dcols[:, idx].reshape(C, k, k)
                    x_pad[:, i*self.stride:i*self.stride+k, j*self.stride:j*self.stride+k] += patch_grad
                    idx += 1

            if self.padding > 0:
                dx[n] = x_pad[:, self.padding:-self.padding, self.padding:-self.padding]
            else:
                dx[n] = x_pad

        self.dW = dW
        self.db = db
        return dx

In [ ]:
class ReLU:
    def forward(self, x):
        self.mask = (x > 0)
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask


class MaxPool2D:
    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride

    def forward(self, x):
        # x: (N, C, H, W)
        self.x = x
        N, C, H, W = x.shape
        p = self.pool_size
        s = self.stride

        out_h = (H - p) // s + 1
        out_w = (W - p) // s + 1

        out = np.zeros((N, C, out_h, out_w))
        self.argmax = np.zeros((N, C, out_h, out_w), dtype=int)

        for n in range(N):
            for c in range(C):
                for i in range(out_h):
                    for j in range(out_w):
                        window = x[n, c, i*s:i*s+p, j*s:j*s+p]
                        out[n, c, i, j] = np.max(window)
                        self.argmax[n, c, i, j] = np.argmax(window)

        return out

    def backward(self, dout):
        N, C, out_h, out_w = dout.shape
        p = self.pool_size
        s = self.stride

        dx = np.zeros_like(self.x)

        for n in range(N):
            for c in range(C):
                for i in range(out_h):
                    for j in range(out_w):
                        idx = self.argmax[n, c, i, j]
                        r, col = divmod(idx, p)
                        dx[n, c, i*s+r, j*s+col] += dout[n, c, i, j]

        return dx


class Flatten:
    def forward(self, x):
        self.input_shape = x.shape
        return x.reshape(x.shape[0], -1)

    def backward(self, dout):
        return dout.reshape(self.input_shape)


class Linear:
    def __init__(self, in_dim, out_dim, weight_scale=0.1):
        self.W = np.random.randn(in_dim, out_dim) * weight_scale
        self.b = np.zeros(out_dim)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        self.dW = self.x.T @ dout
        self.db = np.sum(dout, axis=0)
        dx = dout @ self.W.T
        return dx

In [ ]:
class SoftmaxCrossEntropy:
    def forward(self, logits, y_onehot):
        self.y_onehot = y_onehot
        self.probs = softmax(logits)
        return cross_entropy_loss(self.probs, y_onehot)

    def backward(self):
        N = self.y_onehot.shape[0]
        return (self.probs - self.y_onehot) / N

In [ ]:
class SmallManualCNN:
    def __init__(self):
        self.conv = Conv2D(1, 4, kernel_size=3, stride=1, padding=1, weight_scale=0.1)
        self.relu = ReLU()
        self.pool = MaxPool2D(pool_size=2, stride=2)
        self.flatten = Flatten()
        self.fc = Linear(4 * 4 * 4, 10, weight_scale=0.1)
        self.loss_fn = SoftmaxCrossEntropy()

        self.params = [self.conv, self.fc]

    def forward(self, x, y_onehot=None):
        z = self.conv.forward(x)
        z = self.relu.forward(z)
        z = self.pool.forward(z)
        z = self.flatten.forward(z)
        logits = self.fc.forward(z)

        if y_onehot is None:
            return logits

        loss = self.loss_fn.forward(logits, y_onehot)
        return logits, loss

    def backward(self):
        dout = self.loss_fn.backward()
        dout = self.fc.backward(dout)
        dout = self.flatten.backward(dout)
        dout = self.pool.backward(dout)
        dout = self.relu.backward(dout)
        dout = self.conv.backward(dout)

    def step(self, lr=0.05):
        for layer in self.params:
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db

In [ ]:
def iterate_minibatches(X, y_onehot, y, batch_size=64, shuffle=True):
    idx = np.arange(len(X))
    if shuffle:
        np.random.shuffle(idx)

    for start in range(0, len(X), batch_size):
        batch_idx = idx[start:start+batch_size]
        yield X[batch_idx], y_onehot[batch_idx], y[batch_idx]

In [ ]:
model = SmallManualCNN()

history = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": [],
}

LR = 0.05
EPOCHS = 100
BATCH_SIZE = 64

for epoch in range(1, EPOCHS + 1):
    batch_losses = []
    batch_accs = []

    for xb, yb_onehot, yb in iterate_minibatches(X_train, y_train_onehot, y_train, batch_size=BATCH_SIZE):
        logits, loss = model.forward(xb, yb_onehot)
        acc = accuracy_from_logits(logits, yb)

        model.backward()
        model.step(lr=LR)

        batch_losses.append(loss)
        batch_accs.append(acc)

    train_loss = np.mean(batch_losses)
    train_acc = np.mean(batch_accs)

    test_logits, test_loss = model.forward(X_test, y_test_onehot)
    test_acc = accuracy_from_logits(test_logits, y_test)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"test_loss={test_loss:.4f} test_acc={test_acc:.4f}"
        )

In [ ]:
epochs = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], label="train")
plt.plot(epochs, history["test_loss"], label="test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss Curve")

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], label="train")
plt.plot(epochs, history["test_acc"], label="test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy Curve")

plt.tight_layout()
plt.show()

In [ ]:
test_logits = model.forward(X_test)
test_pred = np.argmax(test_logits, axis=1)

print("Test Accuracy:", accuracy_score(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
kernels = model.conv.W  # (4,1,3,3)

plt.figure(figsize=(8, 2))
for i in range(kernels.shape[0]):
    plt.subplot(1, kernels.shape[0], i + 1)
    plt.imshow(kernels[i, 0], cmap="gray")
    plt.title(f"K{i}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
sample = X_test[:1]

conv_out = model.conv.forward(sample)
relu_out = model.relu.forward(conv_out)
pool_out = model.pool.forward(relu_out)

print("input shape:", sample.shape)
print("conv  shape:", conv_out.shape)
print("pool  shape:", pool_out.shape)

plt.figure(figsize=(10, 2.5))
plt.subplot(1, 5, 1)
plt.imshow(sample[0, 0], cmap="gray")
plt.title("Input")
plt.axis("off")

for i in range(4):
    plt.subplot(1, 5, i + 2)
    plt.imshow(conv_out[0, i], cmap="gray")
    plt.title(f"F{i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Save all results (data + figures)
# =========================
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

FIG_DIR = "../results/figures/g0_manual_cnn"
TABLE_DIR = "../results/tables"
LOG_DIR = "../results/logs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# =========================
# 1. 保存训练历史
# =========================
history_df = pd.DataFrame({
    "epoch": np.arange(1, EPOCHS + 1),
    "train_loss": history["train_loss"],
    "train_acc": history["train_acc"],
    "test_loss": history["test_loss"],
    "test_acc": history["test_acc"],
})

history_path = os.path.join(TABLE_DIR, "g0_manual_cnn_history.csv")
history_df.to_csv(history_path, index=False)

print("Saved history:", history_path)


# =========================
# 2. 保存最终指标
# =========================
test_logits = model.forward(X_test)
test_pred = np.argmax(test_logits, axis=1)

metrics = {
    "test_accuracy": float(np.mean(test_pred == y_test)),
}

metrics_path = os.path.join(LOG_DIR, "g0_manual_cnn_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print("Saved metrics:", metrics_path)


# =========================
# 3. 保存 classification report
# =========================
report = classification_report(y_test, test_pred, digits=4)

report_path = os.path.join(LOG_DIR, "g0_manual_cnn_report.txt")
with open(report_path, "w") as f:
    f.write(report)

print("Saved report:", report_path)


# =========================
# 4. 保存混淆矩阵（图 + 数据）
# =========================
cm = confusion_matrix(y_test, test_pred)

# 保存数值
cm_path = os.path.join(TABLE_DIR, "g0_manual_cnn_confusion_matrix.csv")
pd.DataFrame(cm).to_csv(cm_path, index=False)

# 保存图
plt.figure(figsize=(6, 5))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "g0_confusion_matrix.png")
plt.savefig(fig_path, dpi=200)
plt.close()

print("Saved confusion matrix:", fig_path)


# =========================
# 5. 保存训练曲线
# =========================
epochs_axis = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_axis, history["train_loss"], label="train")
plt.plot(epochs_axis, history["test_loss"], label="test")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_axis, history["train_acc"], label="train")
plt.plot(epochs_axis, history["test_acc"], label="test")
plt.title("Accuracy")
plt.legend()

plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "g0_training_curves.png")
plt.savefig(fig_path, dpi=200)
plt.close()

print("Saved curves:", fig_path)


# =========================
# 6. 保存卷积核
# =========================
kernels = model.conv.W  # (F,1,3,3)

plt.figure(figsize=(8, 2))
for i in range(kernels.shape[0]):
    plt.subplot(1, kernels.shape[0], i + 1)
    plt.imshow(kernels[i, 0], cmap="gray")
    plt.axis("off")

plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "g0_conv_kernels.png")
plt.savefig(fig_path, dpi=200)
plt.close()

print("Saved kernels:", fig_path)


# =========================
# 7. 保存一个样本的特征图
# =========================
sample = X_test[:1]

conv_out = model.conv.forward(sample)

plt.figure(figsize=(10, 2.5))
plt.subplot(1, kernels.shape[0] + 1, 1)
plt.imshow(sample[0, 0], cmap="gray")
plt.axis("off")

for i in range(kernels.shape[0]):
    plt.subplot(1, kernels.shape[0] + 1, i + 2)
    plt.imshow(conv_out[0, i], cmap="gray")
    plt.axis("off")

plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "g0_feature_maps.png")
plt.savefig(fig_path, dpi=200)
plt.close()

print("Saved feature maps:", fig_path)


print("\nAll results saved successfully.")